# 02 - Ingesta Bronze: TRM (Tasa de Cambio Representativa del Mercado)

Trabajo Práctico 1 - Sección 3 (Fuente 2): Pipeline de ingesta PySpark con **carga incremental obligatoria**

## Fuente

* **Dataset**: Tasa de Cambio Representativa del Mercado - Histórico
* **API**: `https://www.datos.gov.co/resource/mcec-87by.json` (Socrata / SoQL)
* **Volumen en la fuente**: ~8.326 registros históricos (serie diaria) + 1 nuevo registro por día hábil
* **Destino**: `Datos_Empresas.bronze.DE_Semiestructurado_TasaCambio_Api`

## Estrategia de carga incremental

A diferencia de RUES (por mes con `replaceWhere`), TRM es una **serie temporal que crece un registro por día**. El pipeline:

1. Revisa si la tabla Bronze ya existe y, de ser así, obtiene la fecha máxima (`vigenciadesde`) ya cargada.
2. Consulta la API solo por registros **posteriores** a esa fecha (`$where=vigenciadesde > 'ultima_fecha'`).
3. Si la tabla **no existe** (primera ejecución), la carga inicial se limita al **último año** (`$where=vigenciadesde >= 'hoy - 365 días'`), en vez de traer todo el histórico desde 1991 — no se necesita más para demostrar la carga incremental.
4. Escribe siempre con `mode("append")`, de modo que el histórico nunca se reprocesa ni se duplica.

Ejecutar este notebook varias veces en distintos días demuestra que solo se agregan los registros nuevos (0 filas nuevas si se corre dos veces el mismo día, 1+ fila si ya pasó un día hábil desde la última ejecución).

## Regla de inmutabilidad (Capa Bronze)

* Se conservan los nombres originales de columna (`valor`, `unidad`, `vigenciadesde`, `vigenciahasta`).
* No se hace *casting* manual de tipos (la limpieza/tipado fuerte es tarea de la Capa Silver).
* Solo se agregan `_ingested_at` y `_source` como columnas de auditoría.

In [ ]:
%python
import requests
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql import functions as F

BASE_URL = "https://www.datos.gov.co/resource/mcec-87by.json"
LIMIT = 50000
TABLA_DESTINO = "Datos_Empresas.bronze.DE_Semiestructurado_TasaCambio_Api"

# Si la tabla no existe aún, la carga inicial se limita a este número de días hacia atrás
DIAS_CARGA_INICIAL = 365

print(f"Fuente: {BASE_URL}")
print(f"Destino: {TABLA_DESTINO}")

---

## Paso 1: Determinar desde qué fecha traer datos nuevos

In [ ]:
%python
def obtener_ultima_fecha_cargada(tabla):
    """Devuelve la fecha vigenciadesde más reciente ya cargada, o None si la tabla no existe."""
    if not spark.catalog.tableExists(tabla):
        return None

    fila = spark.table(tabla).agg(F.max("vigenciadesde").alias("max_fecha")).collect()[0]
    return fila["max_fecha"]


ultima_fecha = obtener_ultima_fecha_cargada(TABLA_DESTINO)

if ultima_fecha is None:
    # Carga inicial: solo el último año, no todo el histórico desde 1991
    fecha_inicio = (datetime.now() - timedelta(days=DIAS_CARGA_INICIAL)).strftime("%Y-%m-%d")
    WHERE_CLAUSE = f"vigenciadesde >= '{fecha_inicio}'"
    print(f"La tabla Bronze aún no existe: carga inicial limitada al último año (desde {fecha_inicio}).")
    print(f"Filtro: {WHERE_CLAUSE}")
else:
    WHERE_CLAUSE = f"vigenciadesde > '{ultima_fecha}'"
    print(f"Última fecha ya cargada: {ultima_fecha}")
    print(f"Filtro incremental: {WHERE_CLAUSE}")

---

## Paso 2: Descargar solo los registros nuevos

In [ ]:
%python
def descargar_datos(where=None):
    registros = []
    offset = 0

    while True:
        params = {"$limit": LIMIT, "$offset": offset, "$order": "vigenciadesde"}
        if where:
            params["$where"] = where
        resp = requests.get(BASE_URL, params=params)
        resp.raise_for_status()
        pagina = resp.json()

        if not pagina:
            break

        registros.extend(pagina)
        offset += LIMIT

    return registros


registros = descargar_datos(WHERE_CLAUSE)
print(f"Registros nuevos obtenidos de la API: {len(registros)}")

if registros:
    df_pandas = pd.DataFrame(registros)
    print(df_pandas.head())
else:
    df_pandas = None
    print("No hay registros nuevos que cargar (la tabla ya está al día).")

---

## Paso 3: Agregar columnas de auditoría y hacer append en Delta Lake

Esta celda solo escribe si hay registros nuevos, para no ejecutar un `append` vacío innecesariamente.

In [ ]:
%python
if df_pandas is not None:
    df_spark = spark.createDataFrame(df_pandas)

    df_bronze = (
        df_spark
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source", F.lit(BASE_URL))
    )

    (
        df_bronze.write
        .format("delta")
        .mode("append")
        .saveAsTable(TABLA_DESTINO)
    )

    print(f"Se agregaron {df_bronze.count()} registros nuevos a {TABLA_DESTINO} (append, sin reprocesar histórico).")
else:
    print("Nada que escribir en esta ejecución.")

---

## Paso 4: Verificar la carga incremental

La columna `_ingested_at` permite ver, agrupando por fecha de carga, cuántos registros trajo cada ejecución del notebook — evidencia de que el histórico no se reprocesa.

In [ ]:
%sql
SELECT COUNT(*) AS total_filas, MIN(vigenciadesde) AS fecha_mas_antigua, MAX(vigenciadesde) AS fecha_mas_reciente
FROM Datos_Empresas.bronze.DE_Semiestructurado_TasaCambio_Api;

In [ ]:
%sql
-- Registros agregados por cada corrida del pipeline (evidencia de carga incremental)
SELECT DATE(_ingested_at) AS fecha_de_ejecucion, COUNT(*) AS registros_agregados
FROM Datos_Empresas.bronze.DE_Semiestructurado_TasaCambio_Api
GROUP BY DATE(_ingested_at)
ORDER BY fecha_de_ejecucion;

---

## Siguiente paso

Continuar con [`03_Ingesta_Bronze_CIIU.ipynb`](03_Ingesta_Bronze_CIIU.ipynb) para la tercera fuente (catálogo de actividades económicas).